#Dimension Table - Weather

> ## 1. View Data First

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
df = spark.table("himalaya.silver.weather")

> ##2. Drop Ingestion

  _not needed in gold_

In [0]:
df = df.drop("ingested_at")

>##3. Aggregate date from hourly to daily
_it is hourly and it is excessive_

In [0]:
df = df.groupBy("peakid", "date").agg(
    F.round(F.avg("temperature"), 1).alias("temperature"),
    F.round(F.sum("rain"), 2).alias("rain"),
    F.round(F.sum("snowfall"), 2).alias("snowfall"),
    F.round(F.avg("snow_depth"), 2).alias("snow_depth"),
    F.round(F.avg("wind_speed"), 1).alias("wind_speed"),
    F.max("weather_code").alias("weather_code")
)

>##4. Generate Weather Id

In [0]:
df = df.withColumn("weather_id", F.concat_ws("_", F.col("peakid"), F.col("date").cast("string")))

>##5. Weather Code --> weather condition
_same thing but english_

In [0]:
df = df.withColumn("weather_condition", 
    F.when(F.col("weather_code") == 0, "Clear Sky")
     .when(F.col("weather_code").between(1, 3), "Partly Cloudy")
     .when(F.col("weather_code").between(45, 48), "Fog")
     .when(F.col("weather_code").between(51, 67), "Rain")
     .when(F.col("weather_code").between(71, 77), "Snow")
     .when(F.col("weather_code").between(80, 82), "Rain Showers")
     .when(F.col("weather_code").between(85, 86), "Snow Showers")
     .when(F.col("weather_code").between(95, 99), "Thunderstorm")
     .otherwise("Unknown")
)

>##6. Reorder

In [0]:
df = df.select("weather_id", "peakid", "date", "temperature", "rain", "snowfall", "snow_depth", "wind_speed", "weather_condition")

>##7. Write to Gold

In [0]:
GOLD_TABLE = "himalaya.gold.dim_weather"
if not spark.catalog.tableExists(GOLD_TABLE):
    df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
else:
    gold_delta = DeltaTable.forName(spark, GOLD_TABLE)
    gold_delta.alias("gold").merge(
        df.alias("silver"),
        "gold.weather_id = silver.weather_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f"✅ Written to {GOLD_TABLE}")

In [0]:
display(spark.table(GOLD_TABLE).limit(5))